# Modelos: qué calcula una RNN, una LSTM y una GRU

**Finalidad**: seguir una ventana real por dentro de cada celda, y entender la salida, la pérdida y la corrección del intervalo.
**Flujo**: Entrada → Celda paso a paso → Parámetros → Salida en cuantiles → Pérdida → Intervalo
**Requiere**: `make prepare`. Las redes de este cuaderno tienen pesos aleatorios: no se entrena nada.
**Decisiones que ilustra**: apartados 6 y 9 de `docs/DECISIONES.md`.

In [ ]:
import json

import numpy as np
import pandas as pd
import torch

from oil_rnn import explore as ex
from oil_rnn import explore_plots as plots
from oil_rnn.config import load_config
from oil_rnn.data import load_split
from oil_rnn.features import Preprocessor
from oil_rnn.model import RNNRegressor
from oil_rnn.paths import SPLITS_MANIFEST

cfg = load_config()
time_col, target = cfg["data"]["time_column"], cfg["data"]["target"]
manifest = json.loads(SPLITS_MANIFEST.read_text(encoding="utf-8"))


train = load_split("train")
prep = Preprocessor(cfg).fit(train)
win = ex.window_at(prep, train, "2008-07-03")
x = win["X"].to_numpy().copy()
print(f"Una ventana: {x.shape[0]} días × {x.shape[1]} entradas")

## 1. La idea común: un estado que se actualiza día a día

Las tres redes leen la ventana de izquierda a derecha. En cada día combinan la entrada `x` de ese día con su estado `h` anterior y producen un estado nuevo. Tras el día 60, el último estado pasa a una capa lineal que da la predicción.

**RNN simple**: `h = tanh(W·x + U·h)`. Un solo paso; el pasado lejano se diluye.

**LSTM**: añade una memoria `c` y tres puertas con valores entre 0 y 1:
- `f` (olvido): cuánto de la memoria anterior se conserva.
- `i` (entrada): cuánto del candidato nuevo `g` se escribe.
- `o` (salida): cuánto de la memoria se muestra como estado.
- `c = f·c + i·g` y `h = o·tanh(c)`.

**GRU**: dos puertas y sin memoria aparte:
- `r` (reinicio): cuánto del estado anterior entra en el candidato `n`.
- `z` (actualización): cuánto del estado anterior se mantiene.
- `h = (1−z)·n + z·h`.

`ex.cell_trace` aplica estas ecuaciones en NumPy. Primero se comprueba que da lo mismo que PyTorch.

In [ ]:
torch.manual_seed(0)
redes = {
    c: RNNRegressor(prep.n_features, 8, 1, 0.0, cell=c, n_outputs=3) for c in ("rnn", "lstm", "gru")
}
trazas = {}
for celda, red in redes.items():
    trazas[celda] = ex.cell_trace(celda, ex.cell_weights(red.rnn), x)
    with torch.no_grad():
        salida, _ = red.rnn(torch.as_tensor(x, dtype=torch.float32)[None])
    diferencia = np.abs(trazas[celda]["h"] - salida[0].numpy()).max()
    print(f"{celda.upper():<5} diferencia máxima con PyTorch: {diferencia:.1e}")

## 2. Por dentro de cada celda

Cada fila es una unidad oculta (8 en este ejemplo); cada columna, un día de la ventana. Los pesos son aleatorios: las figuras enseñan el mecanismo, no lo aprendido.

In [ ]:
plots.gates_heatmap(trazas["rnn"], "rnn")

In [ ]:
plots.gates_heatmap(trazas["lstm"], "lstm")

In [ ]:
plots.gates_heatmap(trazas["gru"], "gru")

## 3. Parámetros: por qué la LSTM pesa 4 veces más que la RNN

Cada bloque tiene `oculto × (entradas + oculto + 2)` parámetros. La RNN tiene 1 bloque; la GRU, 3; la LSTM, 4. La fórmula se contrasta con los runs reales.

In [ ]:
tamaños = cfg["sweep"]["hidden_size"]
parametros = pd.DataFrame(
    {
        c: [ex.count_params(c, prep.n_features, n, 1, 3) for n in tamaños]
        for c in ("rnn", "lstm", "gru")
    },
    index=pd.Index(tamaños, name="tamaño oculto"),
)
display(parametros)
plots.params_lines(parametros)

In [ ]:
runs = ex.runs_table()
finales = runs[runs["prefix"] == "final"].sort_values("horizon")
finales.assign(
    formula=[
        ex.count_params(r.cell, prep.n_features, r.hidden_size, r.num_layers, 3)
        for r in finales.itertuples()
    ]
)[["horizon", "cell", "hidden_size", "num_layers", "n_params", "formula", "train_seconds"]]

## 4. Salida: tres cuantiles que no se cruzan

La capa final da 3 números. El primero es el cuantil 0,1. Los otros dos son incrementos positivos (`softplus`) que se suman: así 0,1 ≤ 0,5 ≤ 0,9 siempre.

In [ ]:
z = torch.tensor([[-0.4, -1.0, 0.3]])  # lo que sale de la capa lineal
pasos = torch.cat([z[:, :1], torch.nn.functional.softplus(z[:, 1:])], dim=1)
cuantiles = torch.cumsum(pasos, dim=1)
print("capa lineal:      ", z.numpy().round(3))
print("incrementos:      ", pasos.numpy().round(3))
print("cuantiles (y est.):", cuantiles.numpy().round(3))

rendimiento = cuantiles.numpy() * prep.y_scale + prep.y_mean
print("rendimiento a h:  ", rendimiento.round(4))
print(
    "precio en $:      ",
    (win["y_last"] * np.exp(rendimiento)).round(2),
    f"(último conocido {win['y_last']:.2f})",
)

## 5. Pérdida *pinball*: cómo se aprende un cuantil

Para el cuantil 0,9, quedarse corto cuesta 9 veces más que pasarse. El mínimo se alcanza cuando el 90 % de los valores reales queda por debajo. Para 0,5 la pérdida es la mitad del error absoluto: la red aprende la mediana, no la media (decisión 6.3).

In [ ]:
plots.pinball_curves()

## 6. Intervalo: la corrección conforme

Los cuantiles aprendidos cubren el 80 % en train, no necesariamente fuera. Con las predicciones de los bloques de validación se mide cuánto se queda corto el intervalo y se ensancha en esa proporción.

Mediana de las 108 configuraciones de la rejilla por horizonte, antes y después:

In [ ]:
rejilla = runs[runs["prefix"] == "g1"]
cobertura = rejilla.groupby("horizon")[["cv_coverage", "cv_coverage_cqr"]].median()
cobertura.columns = ["Sin corregir", "Corregido"]
plots.coverage_bars(cobertura)

In [ ]:
from oil_rnn import conformal

elegido = ex.same_config(rejilla, finales[finales["horizon"] == 63].iloc[0]).iloc[0]
pred = ex.cv_predictions(elegido["run_id"])
s = conformal.scores(pred["y_true"], pred["y_lo"], pred["y_hi"], "log_return")
q = conformal.correction(s, 0.8)
print(f"Puntuaciones: {len(s)} · fuera del intervalo (s > 0): {(s > 0).mean():.0%}")
print(f"Corrección: {q:+.4f} en log → cada borde se aleja un {np.expm1(q):.1%}")
lo, hi = conformal.apply(pred["y_lo"], pred["y_pred"], pred["y_hi"], q, "log_return")
dentro = ((lo <= pred["y_true"]) & (pred["y_true"] <= hi)).mean()
print(f"Cobertura en esos mismos bloques tras corregir: {dentro:.0%}")

La última cifra es optimista: corrige y mide con los mismos datos. La cobertura honrada es `cv_coverage_cqr`, que corrige cada pliegue solo con los anteriores (decisión 9.1). Su límite: si la volatilidad cambia, como en 2026, la corrección se queda corta (decisión 9.2).

## 7. Resumen: lo que hay que saber decir

- Las tres redes hacen lo mismo: resumir 60 días en un vector. Cambia cómo actualizan ese vector.
- LSTM y GRU usan puertas para decidir qué conservar; la RNN no puede.
- Más parámetros no dio mejores resultados aquí: las tres rinden casi igual (`05_resultados.ipynb`).
- El intervalo sale de la red y se corrige con datos que la red no vio.

Siguiente cuaderno: `05_resultados.ipynb`.